# Lab 11: Planner-Coder-Verifier Loop (DS-STAR Core)

**Navigation** : [Lab 10 <<](Lab10-File-Analyzer.ipynb) | [Index](../../README.md) | [>> Lab 12](Lab12-DS-Star-Workshop.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter la boucle itérative Planner-Coder-Verifier de DS-STAR
2. Créer un système multi-agents pour l'analyse de données
3. Gérer les échecs et raffinements automatiques
4. Orchestrer plusieurs composants LLM en pipeline

### Prérequis
- Lab 10 (File Analyzer) complété
- Connaissance des patterns d'agents
- Configuration multi-provider active

### Durée estimée : 45-60 minutes

## 1. Architecture Planner-Coder-Verifier
```
Question --> [PLANNER] --> Plan
                      |
                      v
                [CODER] --> Code
                      |
                      v
                [EXECUTOR] --> Result
                      |
                      v
                [VERIFIER] --> Success/Retry
```

## 2. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import pandas as pd
import numpy as np
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass
from enum import Enum

from config import get_settings
from utils import LLMClient

ModuleNotFoundError: No module named 'config'

Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

NameError: name 'get_settings' is not defined

## 3. Data Classes

In [3]:
@dataclass
class Plan:
    steps: List[str]
    reasoning: str

@dataclass
class ExecutionResult:
    success: bool
    output: str
    error: Optional[str] = None
    code: Optional[str] = None

class VerificationStatus(Enum):
    SUCCESS = 'success'
    NEEDS_REFINEMENT = 'needs_refinement'
    FAILED = 'failed'

## 4. Planner Agent

In [4]:
class Planner:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def create_plan(self, question: str, context: str) -> Plan:
        prompt = f"""Tu es un planificateur d'analyse de donnees.

CONTEXTE DES FICHIERS:
{context}

QUESTION: {question}

Genere un plan d'analyse en 3-5 etapes. Pour chaque etape, decris:
1. L'objectif
2. La methode
3. Le resultat attendu

Format de sortie:
REASONING: [ton raisonnement]
STEPS:
1. [etape 1]
2. [etape 2]
...

Plan:"""

        response = self.llm.generate(prompt, temperature=0.3)

        # Parse response
        steps = []
        reasoning = ""
        lines = response.split('\n')
        in_steps = False

        for line in lines:
            if line.startswith('REASONING:'):
                reasoning = line.replace('REASONING:', '').strip()
            elif line.startswith('STEPS:'):
                in_steps = True
            elif in_steps and re.match(r'^\d+\.', line.strip()):
                steps.append(re.sub(r'^\d+\.\s*', '', line.strip()))

        return Plan(steps=steps, reasoning=reasoning)

NameError: name 'LLMClient' is not defined

## 5. Coder Agent

In [5]:
class Coder:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_code(self, plan: Plan, context: str) -> str:
        steps_text = '\n'.join(f"{i+1}. {s}" for i, s in enumerate(plan.steps))

        prompt = f"""Tu es un programmeur Python expert. Genere du code pour executer ce plan.

CONTEXTE:
{context}

PLAN:
{steps_text}

Genere UNIQUEMENT du code Python executable entre balises ```python ... ```
Le DataFrame est disponible dans la variable 'df'.
Utilise print() pour afficher les resultats.

Code:"""

        response = self.llm.generate(prompt, temperature=0.2)

        # Extract code
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        if match:
            return match.group(1).strip()
        return response

NameError: name 'LLMClient' is not defined

## 6. Executor

In [6]:
class Executor:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.namespace = {'df': df, 'pd': pd, 'np': np, 'print': print}

    def execute(self, code: str) -> ExecutionResult:
        import sys
        from io import StringIO

        old_stdout = sys.stdout
        sys.stdout = StringIO()

        try:
            exec(code, self.namespace)
            output = sys.stdout.getvalue()
            return ExecutionResult(success=True, output=output, code=code)
        except Exception as e:
            output = sys.stdout.getvalue()
            return ExecutionResult(success=False, output=output, error=str(e), code=code)
        finally:
            sys.stdout = old_stdout

## 7. Verifier Agent

In [7]:
class Verifier:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def verify(self, question: str, result: ExecutionResult) -> Tuple[VerificationStatus, str]:
        if not result.success:
            return VerificationStatus.FAILED, f"Erreur d'execution: {result.error}"

        prompt = f"""Verifie si ce resultat repond a la question.

QUESTION: {question}

RESULTAT:
{result.output[:1000]}

Reponds par:
- SUCCESS si le resultat est complet et correct
- NEEDS_REFINEMENT si le resultat est partiel mais prometteur
- FAILED si le resultat ne repond pas du tout

Verdict:"""

        response = self.llm.generate(prompt, temperature=0.1).upper()

        if 'SUCCESS' in response:
            return VerificationStatus.SUCCESS, "Resultat valide"
        elif 'REFINEMENT' in response:
            return VerificationStatus.NEEDS_REFINEMENT, "Necessite des ajustements"
        return VerificationStatus.FAILED, "Resultat insuffisant"

NameError: name 'LLMClient' is not defined

## 8. DS-STAR Orchestrator

In [8]:
class DSStarAgent:
    def __init__(self, df: pd.DataFrame, max_iterations: int = 3):
        self.llm = LLMClient()
        self.planner = Planner(self.llm)
        self.coder = Coder(self.llm)
        self.executor = Executor(df)
        self.verifier = Verifier(self.llm)
        self.max_iterations = max_iterations

    def analyze(self, question: str, file_context: str = None) -> Dict:
        context = file_context or "DataFrame 'df' disponible"

        for iteration in range(self.max_iterations):
            print(f"\n=== ITERATION {iteration + 1}/{self.max_iterations} ===")

            # Step 1: Plan
            print("[PLANNER] Creation du plan...")
            plan = self.planner.create_plan(question, context)
            print(f"Etapes: {len(plan.steps)}")

            # Step 2: Generate code
            print("[CODER] Generation du code...")
            code = self.coder.generate_code(plan, context)

            # Step 3: Execute
            print("[EXECUTOR] Execution...")
            result = self.executor.execute(code)

            if not result.success:
                print(f"[ERROR] {result.error}")
                context += f"\nErreur precedente: {result.error}\nCorrige le code."
                continue

            print(f"[OUTPUT] {result.output[:200]}...")

            # Step 4: Verify
            print("[VERIFIER] Verification...")
            status, message = self.verifier.verify(question, result)
            print(f"[STATUS] {status.value}: {message}")

            if status == VerificationStatus.SUCCESS:
                return {
                    'success': True,
                    'output': result.output,
                    'code': code,
                    'iterations': iteration + 1
                }

            # Raffinement
            context += f"\nResultat partiel: {result.output[:500]}\nAmeliore l'analyse."

        return {
            'success': False,
            'output': result.output if 'result' in dir() else '',
            'error': 'Max iterations atteint',
            'iterations': self.max_iterations
        }

## 9. Test avec un Dataset

In [9]:
# Dataset de test
import tempfile
import os

df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=200, freq='D'),
    'product': np.random.choice(['A', 'B', 'C', 'D'], 200),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 200),
    'revenue': np.random.uniform(100, 5000, 200).round(2),
    'units': np.random.randint(1, 100, 200)
})
print(f'Dataset: {len(df)} lignes')
df.head()

Dataset: 200 lignes


,date,product,region,revenue,units
0,2024-01-01,D,Sud,3126.02,14
1,2024-01-02,A,Sud,1688.06,83
2,2024-01-03,D,Sud,791.64,35
3,2024-01-04,C,Est,1315.41,4
4,2024-01-05,C,Nord,3713.18,80


Test de l'agent DS-STAR avec une iteration.

In [10]:
# Test de l'agent DS-STAR avec 1 iteration pour rapidite
agent = DSStarAgent(df, max_iterations=1)

question = "Quelle est la region avec le plus grand revenu moyen?"
result = agent.analyze(question)

print("\n" + "="*50)
print("RESULTAT FINAL:")
print("="*50)
if result['success']:
    print(result['output'])
else:
    print(f"Erreur: {result.get('error', 'Inconnu')}")

NameError: name 'LLMClient' is not defined

## 10. Resume du Lab
### Ce que nous avons implemente
1. **Planner**: Decompose la question en etapes
2. **Coder**: Genere le code Python
3. **Executor**: Execute en toute securite
4. **Verifier**: Valide ou demande raffinement
### Points cles
- La boucle iterative permet de corriger les erreurs
- Le verifier assure la qualite des resultats
- Le contexte est enrichi a chaque iteration
### Prochaine etape
- **Lab 12**: Workshop DS-STAR complet avec fichiers reels

## Exercice : Analysez vos propres donnees

En utilisant l'architecture DS-STAR que vous venez d'apprendre, creez un agent capable d'analyser un dataset de votre choix.

### Objectifs
1. Creer un dataset personnalise (ou utiliser un dataset publique)
2. Configurer l'agent DS-STAR avec ce dataset
3. Poser 3 questions d'analyse differentes
4. Observer le comportement iteratif de l'agent


In [11]:
# TODO: Creez votre propre dataset
# Exemples : donnees de ventes, meteo, scores sportifs, etc.
mon_dataset = pd.DataFrame({
    # Definissez vos colonnes ici
})

# TODO: Instanciez l'agent DS-STAR
mon_agent = DSStarAgent(mon_dataset, max_iterations=2)

# TODO: Posez 3 questions d'analyse progressives
question_1 = "..."  # Question simple (agregation)
question_2 = "..."  # Question intermediaire (comparaison)
question_3 = "..."  # Question complexe (analyse temporelle ou correlation)

# TODO: Executez et analysez les resultats
resultat_1 = mon_agent.analyze(question_1)
print(f"Reussite: {resultat_1['success']}, Iterations: {resultat_1['iterations']}")


NameError: name 'LLMClient' is not defined

### Points de reflexion
- Combien d'iterations sont necessaires pour chaque type de question ?
- Quelles erreurs l'agent rencontre-t-il et comment les corrige-t-il ?
- Comment pourriez-vous ameliorer le verifier ?
